### Parse Tree for Nested Equation

### Example

In [3]:
# pip install lark
from lark import Lark

# 1. 문법 정의 (EBNF 스타일)
grammar = """
    ?start: expr
    ?expr: term (("+" | "-") term)*
    ?term: factor (("*" | "/") factor)*
    ?factor: NUMBER -> number
           | "(" expr ")"
    %import common.NUMBER
    %import common.WS
    %ignore WS
"""

# 2. 파서 생성 및 실행
parser = Lark(grammar, start='start')
text = "(3 + 5) * 2"
tree = parser.parse(text)

print(tree.pretty()) # 트리를 시각적으로 출력

term
  expr
    number	3
    number	5
  number	2



#### Prefix Notation

In [4]:
from lark import Lark, Transformer, v_args

# 1. 전위 표기법 문법 정의 (EBNF)
# 연산자(op)가 먼저 오고 그 뒤에 두 개의 표현식(expr)이 오는 구조입니다.
prefix_grammar = """
    ?start: expr

    ?expr: NUMBER        -> number
         | "+" expr expr -> add
         | "-" expr expr -> sub
         | "*" expr expr -> mul
         | "/" expr expr -> div

    %import common.NUMBER
    %import common.WS
    %ignore WS
"""

# 2. 분석된 트리를 처리할 Transformer 클래스
# 문법에서 정의한 화살표(->) 뒤의 이름들이 메서드 명과 매칭됩니다.
class CalculatePrefix(Transformer):
    def number(self, n):
        return float(n[0])

    def add(self, items):
        return items[0] + items[1]

    def sub(self, items):
        return items[0] - items[1]

    def mul(self, items):
        return items[0] * items[1]

    def div(self, items):
        return items[0] / items[1]

# 3. 파서 초기화 (LALR 알고리즘 사용)
parser = Lark(prefix_grammar, parser='lalr', transformer=CalculatePrefix())

# 4. 테스트 실행
def evaluate(expression):
    try:
        result = parser.parse(expression)
        print(f"Input: {expression:20} | Result: {result}")
    except Exception as e:
        print(f"Error parsing '{expression}': {e}")

# 테스트 케이스
if __name__ == "__main__":
    evaluate("+ 1 2")              # 1 + 2 = 3.0
    evaluate("* + 3 4 5")          # (3 + 4) * 5 = 35.0
    evaluate("- / 10 2 3")         # (10 / 2) - 3 = 2.0
    evaluate("+ * 2 3 / 8 4")      # (2 * 3) + (8 / 4) = 8.0

Input: + 1 2                | Result: 3.0
Input: * + 3 4 5            | Result: 35.0
Input: - / 10 2 3           | Result: 2.0
Input: + * 2 3 / 8 4        | Result: 8.0


In [4]:
from lark import Lark, Transformer

# 1. 문법 정의 (전위 표기법)
prefix_grammar = """
    ?start: expr
    ?expr: NUMBER           -> number
         | "+" expr expr    -> add
         | "-" expr expr    -> sub
         | "*" expr expr    -> mul
         | "/" expr expr    -> div
    %import common.NUMBER
    %import common.WS
    %ignore WS
"""

# 2. 분석기 클래스
class PrefixAnalyzer(Transformer):
    def number(self, n):
        # 숫자의 값과 해당 숫자가 가진 역할들을 리스트로 관리
        return [{"val": str(n[0]), "roles": []}]

    def _assign_roles(self, items, role_name):
        # 해당 서브트리(items) 내의 모든 숫자 객체에 역할을 추가
        for item in items:
            if role_name not in item["roles"]:
                item["roles"].append(role_name)
        return items

    def add(self, args):
        # args[0]: 왼쪽 수식의 모든 숫자들, args[1]: 오른쪽 수식의 모든 숫자들
        left = self._assign_roles(args[0], "Augend")
        right = self._assign_roles(args[1], "Addend")
        return left + right

    def sub(self, args):
        left = self._assign_roles(args[0], "Minuend")
        right = self._assign_roles(args[1], "Subtrahend")
        return left + right

    def mul(self, args):
        left = self._assign_roles(args[0], "Multiplicand")
        right = self._assign_roles(args[1], "Multiplier")
        return left + right

    def div(self, args):
        left = self._assign_roles(args[0], "Dividend")
        right = self._assign_roles(args[1], "Divisor")
        return left + right

# 3. 실행 함수
def analyze_prefix(text):
    parser = Lark(prefix_grammar, parser='lalr')
    tree = parser.parse(text)
    result = PrefixAnalyzer().transform(tree)
    
    # 결과 출력 (순서 유지 및 중복 값 허용)
    for item in result:
        val = item["val"]
        # 중복된 역할 제거 후 출력 (예: Dividend가 여러번 추가되는 것 방지)
        roles = ", ".join(dict.fromkeys(item["roles"]))
        print(f"{val}: {roles}")

# 테스트 실행
examples = [
    "+ / 23 45 123",
    "- / + 166 178 2 / + 57 44 2",
    "/ - 842000 + 458202 1158000 842000",
    "/ + + + + + + 47 51 64 47 54 41 37 7"
]

for i, ex in enumerate(examples, 1):
    print(f"example {i})")
    print(f"Input: {ex}")
    analyze_prefix(ex)
    print("-" * 30)

example 1)
Input: + / 23 45 123
23: Dividend, Augend
45: Divisor, Augend
123: Addend
------------------------------
example 2)
Input: - / + 166 178 2 / + 57 44 2
166: Augend, Dividend, Minuend
178: Addend, Dividend, Minuend
2: Divisor, Minuend
57: Augend, Dividend, Subtrahend
44: Addend, Dividend, Subtrahend
2: Divisor, Subtrahend
------------------------------
example 3)
Input: / - 842000 + 458202 1158000 842000
842000: Minuend, Dividend
458202: Augend, Subtrahend, Dividend
1158000: Addend, Subtrahend, Dividend
842000: Divisor
------------------------------
example 4)
Input: / + + + + + + 47 51 64 47 54 41 37 7
47: Augend, Dividend
51: Addend, Augend, Dividend
64: Addend, Augend, Dividend
47: Addend, Augend, Dividend
54: Addend, Augend, Dividend
41: Addend, Augend, Dividend
37: Addend, Dividend
7: Divisor
------------------------------
